# Urban Mobility Analytics - Set A
A city transport department has collected 3 years of bus ridership data across 50 routes. This notebook analyzes peak loads, detects anomalies, and builds an interactive zone map to help planners reduce congestion.

## Task 1: Data Generation and Cleaning
Load, clean, and enrich the ridership dataset. All noise must be resolved before Tasks 2–4.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap
import warnings
warnings.filterwarnings('ignore')

# =========================
# DATA GENERATION (Starter Code)
# =========================
np.random.seed(42)
n = 500
rng = np.random.default_rng(42)

# --- Base data ---
passengers = np.random.randint(200, 5000, n).astype(object)
delay_min = np.random.choice([np.nan, 0, 5, 10, 15, 30], n)
zone = list(np.random.choice(['North','South','East','West','Central'], n))
route_id = list(np.random.choice([f'R{i:02d}' for i in range(1,51)], n))
dates = list(pd.date_range('2022-01-01', periods=n, freq='D'))

# --- NOISE INJECTION (do NOT modify below) ---
idx = rng.choice(n, size=40, replace=False)
for i in idx[:15]: passengers[i] = np.nan # missing values
for i in idx[15:25]: passengers[i] = str(passengers[i] or 0).split('.')[0] + ' pax' # dirty strings
for i in idx[25:32]: zone[i] = zone[i].lower() # case inconsistency
for i in idx[32:37]: route_id[i] = ' ' + route_id[i] + ' ' # whitespace
for i in idx[37:40]: delay_min[i] = -999 # bad sentinel
for i in rng.choice(n, 8): passengers[i] = 99999 # outlier spikes

df = pd.DataFrame({'route_id': route_id, 'date': dates,
 'passengers': passengers, 'delay_min': delay_min, 'zone': zone})

print("NULL COUNTS BEFORE CLEANING:")
print(df.isnull().sum())

ModuleNotFoundError: No module named 'folium'

### Task 1, Subtask 1
Fix 'passengers': coerce to numeric, fill NaN with column median, cap outliers (>3σ) to the 99th percentile.

In [ ]:
df['passengers'] = pd.to_numeric(df['passengers'], errors='coerce')
df['passengers'] = df['passengers'].fillna(df['passengers'].median())

mean_p = df['passengers'].mean()
std_p = df['passengers'].std()
upper_limit = mean_p + 3 * std_p
p99 = df['passengers'].quantile(0.99)
df.loc[df['passengers'] > upper_limit, 'passengers'] = p99

df['passengers'].describe()

### Task 1, Subtask 2
Fix 'delay_min': replace -999 with NaN, fill with route-wise median delay.

In [ ]:
df['delay_min'] = df['delay_min'].replace(-999, np.nan)
df['delay_min'] = df.groupby('route_id')['delay_min'].transform(
    lambda x: x.fillna(x.median())
)
# Fallback for routes that might be completely NaN
df['delay_min'] = df['delay_min'].fillna(df['delay_min'].median())
df['delay_min'].isnull().sum()

### Task 1, Subtask 3
Fix 'zone': strip whitespace, standardise to Title Case (North/South/East/West/Central only).

In [ ]:
df['zone'] = df['zone'].str.strip().str.title()
df['zone'].value_counts()

### Task 1, Subtask 4
Fix 'route_id': strip leading/trailing whitespace.

In [ ]:
df['route_id'] = df['route_id'].str.strip()
print("Sample cleaned route IDs:", df['route_id'].unique()[:5])

### Task 1, Subtask 5
Add peak_load ('High' >3500, 'Medium' 1500–3500, 'Low' otherwise) and day_of_week columns. Export to cleaned_ridership.csv. Print null counts before/after.

In [ ]:
df['peak_load'] = pd.cut(
    df['passengers'],
    bins=[-np.inf, 1500, 3500, np.inf],
    labels=['Low', 'Medium', 'High']
)
df['day_of_week'] = df['date'].dt.day_name()

print("NULL COUNTS AFTER CLEANING:")
print(df.isnull().sum())

df.to_csv("cleaned_ridership.csv", index=False)
print("\ncleaned_ridership.csv saved successfully")
df.head()

## Task 2: Algorithm — Route Efficiency Ranking (DSA)

### Task 2, Subtask 1
From cleaned data, compute per-route avg_passengers and avg_delay_min using groupby.

In [ ]:
route_stats = df.groupby('route_id').agg(
    avg_passengers=('passengers', 'mean'),
    avg_delay=('delay_min', 'mean')
).reset_index()

route_stats.head()

### Task 2, Subtask 2
Build a list of tuples: (route_id, avg_passengers, avg_delay). Implement Merge Sort to rank by avg_passengers descending.

In [ ]:
route_list = []
for _, row in route_stats.iterrows():
    route_list.append((row['route_id'], row['avg_passengers'], row['avg_delay']))

def merge_sort(data, key_index, descending=True):
    if len(data) <= 1:
        return data

    mid = len(data) // 2
    left = merge_sort(data[:mid], key_index, descending)
    right = merge_sort(data[mid:], key_index, descending)

    return merge(left, right, key_index, descending)

def merge(left, right, key_index, descending):
    result = []
    i = 0
    j = 0

    while i < len(left) and j < len(right):
        if descending:
            condition = left[i][key_index] >= right[j][key_index]
        else:
            condition = left[i][key_index] <= right[j][key_index]
            
        if condition:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1

    result.extend(left[i:])
    result.extend(right[j:])
    return result

rank_by_passengers = merge_sort(route_list, 1, descending=True)
print("Top 3 by passengers:")
for row in rank_by_passengers[:3]:
    print(row)

### Task 2, Subtask 3
Compute efficiency_score = avg_passengers / (1 + avg_delay). Re-rank using your Merge Sort on this score.

In [ ]:
efficiency_list = []
for route, avg_passengers, avg_delay in route_list:
    efficiency_score = avg_passengers / (1 + avg_delay)
    efficiency_list.append((route, avg_passengers, avg_delay, efficiency_score))

# Rank by efficiency_score (index 3)
rank_by_efficiency = merge_sort(efficiency_list, 3, descending=True)

### Task 2, Subtask 4
Print Top 5 and Bottom 5 routes by efficiency_score in a formatted table.

In [ ]:
print("TOP 5 ROUTES BY EFFICIENCY")
print(f"{'Route':<10}{'Avg Passengers':<20}{'Avg Delay':<15}{'Efficiency Score'}")
for row in rank_by_efficiency[:5]:
    print(f"{row[0]:<10}{row[1]:<20.2f}{row[2]:<15.2f}{row[3]:.2f}")

print("\nBOTTOM 5 ROUTES BY EFFICIENCY")
print(f"{'Route':<10}{'Avg Passengers':<20}{'Avg Delay':<15}{'Efficiency Score'}")
for row in rank_by_efficiency[-5:]:
    print(f"{row[0]:<10}{row[1]:<20.2f}{row[2]:<15.2f}{row[3]:.2f}")

### Task 2, Subtask 5
Verify sort correctness: assert the output list is strictly non-increasing by efficiency_score.

In [ ]:
for i in range(len(rank_by_efficiency) - 1):
    assert rank_by_efficiency[i][3] >= rank_by_efficiency[i + 1][3], f"Sort error at index {i}"
print("Merge Sort verification passed: list is non-increasing by efficiency_score")

## Task 3: Statistical Visualization (NumPy + Matplotlib)
Produce a 4-panel analytical figure saved as mobility_dashboard.png.

In [ ]:
fig = plt.figure(figsize=(16, 12))

# ----------------------------------------------------
# Task 3, Subtask 1
# Panel 1 — Line chart: daily total passengers over time + 7-day rolling mean overlay.
# ----------------------------------------------------
plt.subplot(2, 2, 1)
daily = df.groupby('date')['passengers'].sum()
rolling = daily.rolling(7).mean()
plt.plot(daily.index, daily.values, label='Daily Total Passengers')
plt.plot(rolling.index, rolling.values, label='7-Day Rolling Mean')
plt.title('Daily Total Passengers Over Time')
plt.xlabel('Date')
plt.ylabel('Passengers')
plt.legend()

# ----------------------------------------------------
# Task 3, Subtask 2
# Panel 2 — Grouped bar: avg passengers by zone and peak_load category.
# ----------------------------------------------------
plt.subplot(2, 2, 2)
zone_peak = df.groupby(['zone', 'peak_load'], observed=True)['passengers'].mean().unstack()
zone_peak.plot(kind='bar', ax=plt.gca())
plt.title('Average Passengers by Zone and Peak Load')
plt.xlabel('Zone')
plt.ylabel('Average Passengers')
plt.legend(title='Peak Load')

# ----------------------------------------------------
# Task 3, Subtask 3
# Panel 3 — Histogram: delay_min distribution with normal-curve overlay (numpy mean/std).
# ----------------------------------------------------
plt.subplot(2, 2, 3)
delays = df['delay_min'].dropna().values
mean_delay = np.mean(delays)
std_delay = np.std(delays)
count, bins, ignored = plt.hist(delays, bins=15, density=True, alpha=0.6)
normal_curve = (1 / (std_delay * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((bins - mean_delay) / std_delay) ** 2)
plt.plot(bins, normal_curve, label='Normal Curve')
plt.title('Delay Distribution')
plt.xlabel('Delay in Minutes')
plt.ylabel('Density')
plt.legend()

# ----------------------------------------------------
# Task 3, Subtask 4
# Panel 4 — Scatter: avg_passengers vs avg_delay per route, colored by zone, top-10 routes labelled.
# ----------------------------------------------------
plt.subplot(2, 2, 4)
route_zone = df.groupby('route_id')['zone'].agg(lambda x: pd.Series.mode(x)[0]).reset_index()
route_plot = route_stats.merge(route_zone, on='route_id')
zones = route_plot['zone'].unique()
for z in zones:
    temp = route_plot[route_plot['zone'] == z]
    plt.scatter(temp['avg_passengers'], temp['avg_delay'], label=z)

top10 = route_plot.sort_values('avg_passengers', ascending=False).head(10)
for _, row in top10.iterrows():
    plt.text(row['avg_passengers'], row['avg_delay'], row['route_id'])

plt.title('Average Passengers vs Average Delay Per Route')
plt.xlabel('Average Passengers')
plt.ylabel('Average Delay in Minutes')
plt.legend()

# ----------------------------------------------------
# Task 3, Subtask 5
# All panels: titles, axis labels with units, legend. Use plt.tight_layout(). Save at 150 DPI.
# ----------------------------------------------------
plt.tight_layout()
plt.savefig("mobility_dashboard.png", dpi=150)
print("mobility_dashboard.png saved successfully")
plt.show()

## Task 4: Interactive Zone Map (Folium)

### Task 4, Subtask 1
Initialise Folium map centred on Delhi [28.6139, 77.2090] at zoom 11.

In [ ]:
ZONE_COORDS = {
    'North': [28.8000, 77.2090],
    'South': [28.4595, 77.0266],
    'East': [28.6139, 77.3910],
    'West': [28.6139, 77.0270],
    'Central': [28.6139, 77.2090],
}

m = folium.Map(location=[28.6139, 77.2090], zoom_start=11)

### Task 4, Subtasks 2 & 3
2. Per zone: CircleMarker, radius = zone_avg_passengers / 200. Color: green <2000, orange 2000–3500, red >3500.
3. Popup: Zone Name, Avg Passengers, Avg Delay (min), Dominant peak_load category.

In [ ]:
zone_stats = df.groupby('zone').agg(
    zone_avg_passengers=('passengers', 'mean'),
    zone_avg_delay=('delay_min', 'mean'),
    dominant_peak_load=('peak_load', lambda x: pd.Series.mode(x)[0])
).reset_index()

marker_count = 0
for _, row in zone_stats.iterrows():
    z = row['zone']
    avg_passengers = row['zone_avg_passengers']
    avg_delay = row['zone_avg_delay']
    dominant_peak = row['dominant_peak_load']

    if avg_passengers < 2000:
        color = 'green'
    elif avg_passengers <= 3500:
        color = 'orange'
    else:
        color = 'red'

    popup_text = f'''
    <b>Zone:</b> {z}<br>
    <b>Avg Passengers:</b> {avg_passengers:.2f}<br>
    <b>Avg Delay:</b> {avg_delay:.2f} min<br>
    <b>Dominant Peak Load:</b> {dominant_peak}
    '''
    
    folium.CircleMarker(
        location=ZONE_COORDS[z],
        radius=avg_passengers / 200,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.6,
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)
    marker_count += 1

### Task 4, Subtask 4
Add HeatMap layer using all data points (jitter lat/lon ±0.02 around zone centre).

In [ ]:
heat_data = []
for _, row in df.iterrows():
    base_lat, base_lon = ZONE_COORDS[row['zone']]
    jitter_lat = base_lat + np.random.uniform(-0.02, 0.02)
    jitter_lon = base_lon + np.random.uniform(-0.02, 0.02)
    heat_data.append([jitter_lat, jitter_lon, row['passengers']])

HeatMap(heat_data).add_to(m)

### Task 4, Subtask 5
Save as city_mobility_map.html. Print total marker count.

In [ ]:
m.save("city_mobility_map.html")
print("city_mobility_map.html saved successfully")
print("Total marker count:", marker_count)
m

## Streamlit Dashboard Implementation
The following cell contains the complete code for our Streamlit dashboard, showing both the raw generated input data and the cleaned output, along with all visualizations and maps. Running this cell will save the code to `app.py`.

In [ ]:
streamlit_code = """
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap
from streamlit_folium import st_folium

st.set_page_config(page_title="Urban Mobility Analytics", layout="wide")

st.title("Urban Mobility Analytics Dashboard")
st.markdown("Analyzing peak loads, detecting anomalies, and visualizing interactive zone maps to help planners reduce congestion.")

@st.cache_data
def generate_data():
    np.random.seed(42)
    n = 500
    rng = np.random.default_rng(42)

    passengers = np.random.randint(200, 5000, n).astype(object)
    delay_min = np.random.choice([np.nan, 0, 5, 10, 15, 30], n)
    zone = list(np.random.choice(['North', 'South', 'East', 'West', 'Central'], n))
    route_id = list(np.random.choice([f'R{i:02d}' for i in range(1, 51)], n))
    dates = list(pd.date_range('2022-01-01', periods=n, freq='D'))

    idx = rng.choice(n, size=40, replace=False)
    for i in idx[:15]: passengers[i] = np.nan
    for i in idx[15:25]: passengers[i] = str(passengers[i] or 0).split('.')[0] + ' pax'
    for i in idx[25:32]: zone[i] = zone[i].lower()
    for i in idx[32:37]: route_id[i] = ' ' + route_id[i] + ' '
    for i in idx[37:40]: delay_min[i] = -999
    for i in rng.choice(n, 8): passengers[i] = 99999

    df = pd.DataFrame({
        'route_id': route_id,
        'date': dates,
        'passengers': passengers,
        'delay_min': delay_min,
        'zone': zone
    })
    return df
    
@st.cache_data
def clean_data(df):
    df = df.copy()
    df['passengers'] = pd.to_numeric(df['passengers'], errors='coerce')
    df['passengers'] = df['passengers'].fillna(df['passengers'].median())

    mean_p = df['passengers'].mean()
    std_p = df['passengers'].std()
    upper_limit = mean_p + 3 * std_p
    p99 = df['passengers'].quantile(0.99)
    df.loc[df['passengers'] > upper_limit, 'passengers'] = p99

    df['delay_min'] = df['delay_min'].replace(-999, np.nan)
    df['route_id'] = df['route_id'].str.strip()

    df['delay_min'] = df.groupby('route_id')['delay_min'].transform(
        lambda x: x.fillna(x.median())
    )
    df['delay_min'] = df['delay_min'].fillna(df['delay_min'].median())
    df['zone'] = df['zone'].str.strip().str.title()

    df['peak_load'] = pd.cut(
        df['passengers'],
        bins=[-np.inf, 1500, 3500, np.inf],
        labels=['Low', 'Medium', 'High']
    )
    df['day_of_week'] = df['date'].dt.day_name()
    return df

raw_df = generate_data()
df = clean_data(raw_df)

st.header("1. Sample Input (Raw Data)")
st.markdown("This is the raw dataset with injected noise and missing values.")
st.write(raw_df.head(10))

st.header("2. Cleaned Data Overview")
st.markdown("This is the dataset after completing the ETL process (handling missing values, standardizing strings, capping outliers).")
st.write(df.head(10))

st.header("3. Route Efficiency Ranking")
def merge_sort(data, key_index):
    if len(data) <= 1: return data
    mid = len(data) // 2
    left = merge_sort(data[:mid], key_index)
    right = merge_sort(data[mid:], key_index)
    return merge(left, right, key_index)

def merge(left, right, key_index):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i][key_index] >= right[j][key_index]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

route_stats = df.groupby('route_id').agg(
    avg_passengers=('passengers', 'mean'),
    avg_delay=('delay_min', 'mean')
).reset_index()

route_list = []
for _, row in route_stats.iterrows():
    route_list.append((row['route_id'], row['avg_passengers'], row['avg_delay']))

efficiency_list = []
for route, avg_passengers, avg_delay in route_list:
    efficiency_score = avg_passengers / (1 + avg_delay)
    efficiency_list.append((route, avg_passengers, avg_delay, efficiency_score))

rank_by_efficiency = merge_sort(efficiency_list, 3)
efficiency_df = pd.DataFrame(rank_by_efficiency, columns=["Route", "Avg Passengers", "Avg Delay", "Efficiency Score"])

col1, col2 = st.columns(2)
with col1:
    st.subheader("Top 5 Routes by Efficiency")
    st.dataframe(efficiency_df.head())
with col2:
    st.subheader("Bottom 5 Routes by Efficiency")
    st.dataframe(efficiency_df.tail())

st.header("4. Statistical Visualization")
fig = plt.figure(figsize=(16, 12))

# Panel 1
plt.subplot(2, 2, 1)
daily = df.groupby('date')['passengers'].sum()
rolling = daily.rolling(7).mean()
plt.plot(daily.index, daily.values, label='Daily Total Passengers')
plt.plot(rolling.index, rolling.values, label='7-Day Rolling Mean')
plt.title('Daily Total Passengers Over Time')
plt.xlabel('Date')
plt.ylabel('Passengers')
plt.legend()

# Panel 2
plt.subplot(2, 2, 2)
zone_peak = df.groupby(['zone', 'peak_load'], observed=True)['passengers'].mean().unstack()
zone_peak.plot(kind='bar', ax=plt.gca())
plt.title('Average Passengers by Zone and Peak Load')
plt.xlabel('Zone')
plt.ylabel('Average Passengers')
plt.legend(title='Peak Load')

# Panel 3
plt.subplot(2, 2, 3)
delays = df['delay_min'].values
mean_delay = np.mean(delays)
std_delay = np.std(delays)
count, bins, ignored = plt.hist(delays, bins=15, density=True, alpha=0.6)
normal_curve = (1 / (std_delay * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((bins - mean_delay) / std_delay) ** 2)
plt.plot(bins, normal_curve, label='Normal Curve')
plt.title('Delay Distribution')
plt.xlabel('Delay in Minutes')
plt.ylabel('Density')
plt.legend()

# Panel 4
plt.subplot(2, 2, 4)
route_zone = df.groupby('route_id')['zone'].agg(lambda x: pd.Series.mode(x)[0]).reset_index()
route_plot = route_stats.merge(route_zone, on='route_id')
zones = route_plot['zone'].unique()
for z in zones:
    temp = route_plot[route_plot['zone'] == z]
    plt.scatter(temp['avg_passengers'], temp['avg_delay'], label=z)

top10 = route_plot.sort_values('avg_passengers', ascending=False).head(10)
for _, row in top10.iterrows():
    plt.text(row['avg_passengers'], row['avg_delay'], row['route_id'])

plt.title('Average Passengers vs Average Delay Per Route')
plt.xlabel('Average Passengers')
plt.ylabel('Average Delay in Minutes')
plt.legend()
plt.tight_layout()

st.pyplot(fig)

st.header("5. Interactive Zone Map")

ZONE_COORDS = {
    'North': [28.8000, 77.2090],
    'South': [28.4595, 77.0266],
    'East': [28.6139, 77.3910],
    'West': [28.6139, 77.0270],
    'Central': [28.6139, 77.2090],
}

m = folium.Map(location=[28.6139, 77.2090], zoom_start=11)
zone_stats = df.groupby('zone').agg(
    zone_avg_passengers=('passengers', 'mean'),
    zone_avg_delay=('delay_min', 'mean'),
    dominant_peak_load=('peak_load', lambda x: pd.Series.mode(x)[0])
).reset_index()

for _, row in zone_stats.iterrows():
    z = row['zone']
    avg_passengers = row['zone_avg_passengers']
    avg_delay = row['zone_avg_delay']
    dominant_peak = row['dominant_peak_load']

    if avg_passengers < 2000:
        color = 'green'
    elif avg_passengers <= 3500:
        color = 'orange'
    else:
        color = 'red'

    popup_text = f"<b>Zone:</b> {z}<br><b>Avg Passengers:</b> {avg_passengers:.2f}<br><b>Avg Delay:</b> {avg_delay:.2f} min<br><b>Dominant Peak Load:</b> {dominant_peak}"
    
    folium.CircleMarker(
        location=ZONE_COORDS[z],
        radius=avg_passengers / 200,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.6,
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)

heat_data = []
for _, row in df.iterrows():
    base_lat, base_lon = ZONE_COORDS[row['zone']]
    jitter_lat = base_lat + np.random.uniform(-0.02, 0.02)
    jitter_lon = base_lon + np.random.uniform(-0.02, 0.02)
    heat_data.append([jitter_lat, jitter_lon, row['passengers']])

HeatMap(heat_data).add_to(m)

st_folium(m, width=1000, height=600)
"""
with open('app.py', 'w', encoding='utf-8') as app_f:
    app_f.write(streamlit_code)
print('app.py has been successfully generated from this notebook!')

## Run the Streamlit Dashboard
Run the cell below to start the Streamlit application from within this notebook!

**Note:** This cell will run continuously to keep the server alive. You will see a local URL (e.g., `http://localhost:8501`) that you can click to view your app. To stop the dashboard, simply stop/interrupt the execution of the cell.

In [ ]:
!python -m streamlit run app.py